# PSR-SRS MVP — End-to-End Pipeline

## E-commerce Personalized Search Ranking & Semantic Retrieval

**MVP Goal**: Demonstrate a complete pipeline from synthetic data generation through
multi-channel retrieval, user profiling, personalized re-ranking, and offline evaluation — all running locally.

## Pipeline Structure

```
Synthetic Data Generation
  → BM25 Keyword Retrieval
  → LSA Semantic Retrieval
  → Linear Hybrid Fusion (0.5/0.5)
  → Personalized Re-ranking (0.70 retrieval + 0.30 affinity)
  → Offline Evaluation (qrels + behavior metrics)
```

## This Notebook Covers

1. Data loading & quality summary
2. Data distribution analysis
3. Train/test split & time-leakage check
4. BM25 keyword retrieval
5. LSA semantic retrieval
6. BM25 vs LSA comparison
7. Linear Hybrid fusion
8. User profiling
9. Personalized re-ranking with case studies
10. Behavior metrics (HitRate, MRR, NDCG, Positive Recall)
11. Qrels protection metrics
12. Candidate positive coverage analysis
13. Fallback verification
14. Full model comparison & conclusions

**Limitation**: All data is synthetic, generated by rule-based simulation.
Results do not represent real-world e-commerce performance.


## 1. Environment & Configuration

In [1]:
import os, sys, json, csv, hashlib
from pathlib import Path
from collections import Counter, defaultdict

# Auto-locate project root (works from project root or notebooks/ dir)
PROJECT_ROOT = Path.cwd()
for _ in range(3):
    if (PROJECT_ROOT / "src" / "psr_srs_mvp").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

_src = str(PROJECT_ROOT / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

SRC = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data" / "sample"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
CONFIG_DIR = PROJECT_ROOT / "configs"

SEED = 20260614
RECOMPUTE = os.getenv("PSR_SRS_RECOMPUTE", "0") == "1"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir:     {DATA_DIR}")
print(f"Config dir:   {CONFIG_DIR}")
print(f"Seed:         {SEED}")
print(f"Recompute:    {RECOMPUTE}  (env PSR_SRS_RECOMPUTE={os.getenv('PSR_SRS_RECOMPUTE', '0')})")
print(f"Python:       {sys.version.split()[0]}")


Project root: D:\project\PSR-SRS-MVP
Data dir:     D:\project\PSR-SRS-MVP\data\sample
Config dir:   D:\project\PSR-SRS-MVP\configs
Seed:         20260614
Recompute:    False  (env PSR_SRS_RECOMPUTE=0)
Python:       3.12.6


## 2. Data Loading

In [2]:
from psr_srs_mvp.retrieval.io import load_items, load_queries, load_qrels
from psr_srs_mvp.personalization.split import load_events
from psr_srs_mvp.personalization.profiles import load_users_map

items_raw = load_items(DATA_DIR / "items.csv")
users_raw = load_users_map(DATA_DIR / "users.csv")
queries_raw = load_queries(DATA_DIR / "queries.csv")
events_raw = load_events(DATA_DIR / "events.csv")
qrels_raw = load_qrels(DATA_DIR / "qrels.csv")

print(f"{'Table':<12} {'Rows':>8} {'Columns':>8}")
print("-" * 30)
print(f"{'items.csv':<12} {len(items_raw):>8} {len(items_raw[0]) if items_raw else 0:>8}")
print(f"{'users.csv':<12} {len(users_raw):>8} {7:>8}")
print(f"{'queries.csv':<12} {len(queries_raw):>8} {len(queries_raw[0]) if queries_raw else 0:>8}")
print(f"{'events.csv':<12} {len(events_raw):>8} {len(events_raw[0]) if events_raw else 0:>8}")
print(f"{'qrels.csv':<12} {len(qrels_raw):>8} {3:>8}")

# Sample rows
print("\n--- Sample: items.csv (first 3 rows) ---")
for r in items_raw[:3]:
    print({k: r[k] for k in ['item_id','title','category','brand','price']})

print("\n--- Sample: users.csv (first 3 rows) ---")
for uid in sorted(users_raw)[:3]:
    r = users_raw[uid]
    print({k: r[k] for k in ['user_id','activity_level','is_cold_start']})

print("\n--- Sample: queries.csv (first 3 rows) ---")
for r in queries_raw[:3]:
    print({k: r[k] for k in ['query_id','query_text','intended_category']})


Table            Rows  Columns
------------------------------
items.csv         500       11
users.csv         100        7
queries.csv       200        5
events.csv       6376       13
qrels.csv         200        3

--- Sample: items.csv (first 3 rows) ---
{'item_id': 'item_000001', 'title': 'EcoFresh Modern Air Quality', 'category': 'Health & Household', 'brand': 'EcoFresh', 'price': '3.38'}
{'item_id': 'item_000002', 'title': 'EcoFresh Ultra Air Quality', 'category': 'Health & Household', 'brand': 'EcoFresh', 'price': '3.63'}
{'item_id': 'item_000003', 'title': 'PeakAscent Deluxe Yoga', 'category': 'Sports & Outdoors', 'brand': 'PeakAscent', 'price': '406.01'}

--- Sample: users.csv (first 3 rows) ---
{'user_id': 'user_000001', 'activity_level': 'low', 'is_cold_start': 'true'}
{'user_id': 'user_000002', 'activity_level': 'low', 'is_cold_start': 'true'}
{'user_id': 'user_000003', 'activity_level': 'low', 'is_cold_start': 'true'}

--- Sample: queries.csv (first 3 rows) ---
{'query_id

## 3. Data Quality Summary

Reading pre-computed quality report and reproducibility report.

In [3]:
import json

# Quality report
qr_path = OUTPUT_DIR / "data_generation" / "data_quality_report.json"
if qr_path.exists():
    qr = json.loads(qr_path.read_text(encoding="utf-8"))
    qrep = qr.get("quality_report", qr)
    print(f"Total checks:  {qrep['total_checks']}")
    print(f"Passed:        {qrep['passed_count']}")
    print(f"Errors:        {qrep['error_count']}")
    print(f"Warnings:      {qrep['warning_count']}")
else:
    print("Quality report not found — run: scripts/validate_data.py")

# Reproducibility report
rr_path = OUTPUT_DIR / "data_generation" / "reproducibility_report.json"
if rr_path.exists():
    rr = json.loads(rr_path.read_text(encoding="utf-8"))
    print(f"\nReproducibility: {'PASSED' if rr['overall_passed'] else 'FAILED'}")
    for fn, info in rr["files"].items():
        print(f"  {fn}: {'MATCH' if info['matched'] else 'DIFFER'}  sha256={info['sha256_run_a'][:16]}...  lines={info['lines_a']}")
else:
    print("\nReproducibility report not found — run: scripts/reproducibility_check.py")

# Manifest
mp_path = OUTPUT_DIR / "data_generation" / "data_quality_report.json"
if mp_path.exists():
    mf = json.loads(mp_path.read_text(encoding="utf-8"))
    if "manifest" in mf:
        print(f"\nManifest files:")
        for fn, fi in mf["manifest"].get("files", {}).items():
            print(f"  {fn}: {fi['rows']} rows, sha256={fi['sha256'][:16]}...")


Total checks:  66
Passed:        66
Errors:        0
Warnings:      0

Reproducibility: PASSED
  items.csv: MATCH  sha256=d5068a93c9bde32a...  lines=501
  users.csv: MATCH  sha256=2569d4eed8d98f85...  lines=101
  queries.csv: MATCH  sha256=cd368caed880a0df...  lines=201
  events.csv: MATCH  sha256=1ed4f332924bd4cf...  lines=6377
  qrels.csv: MATCH  sha256=bf2042bf80381565...  lines=10077

Manifest files:
  items.csv: 500 rows, sha256=d5068a93c9bde32a...
  users.csv: 100 rows, sha256=2569d4eed8d98f85...
  queries.csv: 200 rows, sha256=cd368caed880a0df...
  events.csv: 6376 rows, sha256=1ed4f332924bd4cf...
  qrels.csv: 10076 rows, sha256=bf2042bf80381565...


## 4. Data Distribution Analysis

### User Activity & Event Distribution

In [4]:
from collections import Counter

# User activity_level
act = Counter(u["activity_level"] for u in users_raw.values())
print("User activity_level:")
for k in ["low","medium","high"]:
    print(f"  {k}: {act.get(k,0)}")

# Sessions per user
user_sessions = defaultdict(set)
for e in events_raw:
    user_sessions[e["user_id"]].add(e["session_id"])
sess_counts = [len(s) for s in user_sessions.values()]
multi = sum(1 for c in sess_counts if c >= 2)
single = sum(1 for c in sess_counts if c == 1)
zero = len(users_raw) - len(user_sessions)
print(f"\nUsers by session count:")
print(f"  Multi-session (>=2): {multi}")
print(f"  Single-session:      {single}")
print(f"  Zero-session:        {zero}")
print(f"  Total:               {multi+single+zero}")

# Event type distribution
etypes = Counter(e["event_type"] for e in events_raw)
print(f"\nEvent types:")
for et in ["impression","click","favorite","add_to_cart","purchase"]:
    print(f"  {et:15s}: {etypes.get(et,0):6d}")

# Category distribution
cats = Counter(r["category"] for r in items_raw)
print(f"\nTop item categories:")
for cat, cnt in cats.most_common(5):
    print(f"  {cat}: {cnt}")


User activity_level:
  low: 33
  medium: 30
  high: 37

Users by session count:
  Multi-session (>=2): 65
  Single-session:      22
  Zero-session:        13
  Total:               100

Event types:
  impression     :   5752
  click          :    529
  favorite       :     47
  add_to_cart    :     40
  purchase       :      8

Top item categories:
  Health & Household: 50
  Sports & Outdoors: 50
  Books: 50
  Electronics: 50
  Beauty & Personal Care: 50


## 5. Train/Test Split & Time Leakage Check

In [5]:
from psr_srs_mvp.personalization.split import split_events
from psr_srs_mvp.personalization import PersonalizationConfig
import json

cfg = PersonalizationConfig.from_json(CONFIG_DIR / "personalization.json")
train_evts, test_evts, split_info = split_events(events_raw, cfg.train_ratio)

train_sids = len({e["session_id"] for e in train_evts})
test_sids = len({e["session_id"] for e in test_evts})
all_sids = len({e["session_id"] for e in events_raw})

print(f"Configured sessions:  500")
print(f"Unique in events:     {all_sids}")
print(f"Train sessions:       {train_sids}")
print(f"Test sessions:        {test_sids}")
print(f"Zero-event (config):  {500 - all_sids}")
print(f"Time leakage:         {'PASS' if split_info['time_leakage_free'] else 'FAIL'}")

# Example user timeline
from datetime import datetime
example_uid = sorted(user_sessions.keys())[0]
example_sessions = sorted(user_sessions[example_uid])
session_times = {}
for e in events_raw:
    if e["user_id"] == example_uid:
        sid = e["session_id"]
        ts = datetime.fromisoformat(e["timestamp"])
        session_times[sid] = min(session_times.get(sid, ts), ts)

# Determine which sessions are test
test_sids_set = {e["session_id"] for e in test_evts}
print(f"\nExample user: {example_uid}")
print(f"  Total sessions: {len(example_sessions)}")
for sid in sorted(example_sessions, key=lambda s: session_times.get(s, datetime.min)):
    label = "TEST" if sid in test_sids_set else "train"
    print(f"  {sid}: {session_times.get(sid, 'N/A')} [{label}]")

print("\nImage construction uses ONLY train sessions.")
print("Test behavior is NEVER used for training profiles.")


Configured sessions:  500
Unique in events:     484
Train sessions:       400
Test sessions:        84
Zero-event (config):  16
Time leakage:         PASS

Example user: user_000001
  Total sessions: 2
  sess_000355: 2026-01-02 16:40:03.254997+00:00 [train]
  sess_000208: 2026-02-10 17:44:40.989902+00:00 [TEST]

Image construction uses ONLY train sessions.
Test behavior is NEVER used for training profiles.


## 6. BM25 Keyword Retrieval

Loading BM25 baseline results from pre-computed outputs. Set `RECOMPUTE=True` to rebuild index.

In [6]:
from psr_srs_mvp.retrieval import BM25Config, BM25Index, Document, tokenize, build_item_text

bm25_cfg = BM25Config.from_json(CONFIG_DIR / "bm25.json")

if RECOMPUTE:
    bm25_texts = [build_item_text(r["title"], r["description"], r["category"],
                                   r["subcategory"], r["brand"],
                                   weights=bm25_cfg.field_weights) for r in items_raw]
    bm25_docs = [Document(item_id=r["item_id"],
                          tokens=tokenize(bm25_texts[i], remove_stopwords=bm25_cfg.use_stopwords),
                          length=len(tokenize(bm25_texts[i], remove_stopwords=bm25_cfg.use_stopwords)))
                 for i, r in enumerate(items_raw)]
    bm25_idx = BM25Index.build(bm25_docs, k1=bm25_cfg.k1, b=bm25_cfg.b)
else:
    bm25_idx = None  # Use cached results

# Load cached BM25 metrics
bm25_metrics = json.loads((OUTPUT_DIR / "bm25" / "metrics.json").read_text(encoding="utf-8"))
print(f"BM25 NDCG@10: {bm25_metrics.get('ndcg_at_10', 'N/A')}")
print(f"BM25 query coverage from report: ~54% (108/200)")

# Show example query results from cached search_results.csv
bm25_sr = list(csv.DictReader(open(OUTPUT_DIR / "bm25" / "search_results.csv", encoding="utf-8", newline="")))
print(f"\nTotal cached BM25 results: {len(bm25_sr)} rows")

# Show top results for a specific query
example_qid = "query_000001"
example_results = [r for r in bm25_sr if r["query_id"] == example_qid]
print(f"\nExample: {example_qid} -> {example_results[0]['query_text'] if example_results else 'N/A'}")
print(f"{'Rank':<6} {'Item ID':<15} {'BM25 Score':<12} {'Grade':<8}")
for r in example_results[:10]:
    print(f"{r['rank']:<6} {r['item_id']:<15} {float(r['bm25_score']):<12.4f} {r['relevance_grade']:<8}")

# Also show an empty query
empty_qs = [q for q in queries_raw if q["query_id"] not in {r["query_id"] for r in bm25_sr}]
print(f"\nQueries with zero BM25 results: {len(empty_qs)} / {len(queries_raw)}")


BM25 NDCG@10: 0.297994
BM25 query coverage from report: ~54% (108/200)

Total cached BM25 results: 2160 rows

Example: query_000001 -> NeonFit women's dresses review
Rank   Item ID         BM25 Score   Grade   
1      item_000063     12.6591      2       
2      item_000080     12.6591      2       
3      item_000099     12.6591      2       
4      item_000107     12.6591      2       
5      item_000149     12.6591      2       
6      item_000206     12.6591      2       
7      item_000225     12.6591      2       
8      item_000250     12.6591      2       
9      item_000254     12.6591      2       
10     item_000263     12.6591      2       

Queries with zero BM25 results: 92 / 200


## 7. LSA Semantic Retrieval

In [7]:
lsa_metrics = json.loads((OUTPUT_DIR / "semantic" / "metrics.json").read_text(encoding="utf-8"))
print(f"LSA NDCG@10: {lsa_metrics.get('ndcg_at_10', 'N/A')}")
print(f"LSA query coverage: {lsa_metrics.get('query_coverage', 'N/A')}")
print(f"SVD components: {lsa_metrics.get('svd_components_actual', 'N/A')}")
print(f"Combined features: {lsa_metrics.get('combined_feature_count', 'N/A')}")
print(f"Explained variance: {lsa_metrics.get('explained_variance_ratio_sum', 'N/A')}")

# Compare BM25 vs LSA on a query where BM25 fails
from collections import defaultdict
bm25_by_qid = defaultdict(list)
for r in csv.DictReader(open(OUTPUT_DIR / "bm25" / "search_results.csv", encoding="utf-8", newline="")):
    bm25_by_qid[r["query_id"]].append(r)

lsa_sr = list(csv.DictReader(open(OUTPUT_DIR / "semantic" / "search_results.csv", encoding="utf-8", newline="")))
lsa_by_qid = defaultdict(list)
for r in lsa_sr:
    lsa_by_qid[r["query_id"]].append(r)

# Find a query with BM25=0 results but LSA has results
for q in queries_raw:
    qid = q["query_id"]
    if qid not in bm25_by_qid and qid in lsa_by_qid:
        print(f"\nQuery rescued by LSA: {qid} -> '{q['query_text']}'")
        print(f"  BM25 results: 0")
        print(f"  LSA results: {len(lsa_by_qid[qid])}")
        for r in lsa_by_qid[qid][:5]:
            print(f"    rank={r['rank']} item={r['item_id']} score={float(r['semantic_score']):.4f} grade={r['relevance_grade']}")
        break


LSA NDCG@10: 0.37332
LSA query coverage: 0.995
SVD components: 64
Combined features: 1989
Explained variance: 0.933128

Query rescued by LSA: query_000003 -> 'PureComfort Activewear'
  BM25 results: 0
  LSA results: 20
    rank=1 item=item_000326 score=0.7079 grade=1
    rank=2 item=item_000250 score=0.6849 grade=1
    rank=3 item=item_000330 score=0.6849 grade=1
    rank=4 item=item_000396 score=0.6836 grade=1
    rank=5 item=item_000467 score=0.6836 grade=1


## 8. BM25 vs LSA Comparison

In [8]:
comp = json.loads((OUTPUT_DIR / "comparison" / "bm25_vs_semantic.json").read_text(encoding="utf-8"))

print(f"{'Metric':<18} {'BM25':>10} {'LSA':>10} {'Delta':>10}")
print("-" * 50)
for k in ["precision_at_10", "recall_at_20", "mrr_at_10", "ndcg_at_10"]:
    b = comp["bm25"].get(k, 0)
    s = comp["semantic"].get(k, 0)
    d = comp["semantic_minus_bm25"].get(k, 0)
    print(f"{k:<18} {b:>10.6f} {s:>10.6f} {d:>+10.6f}")

print(f"\nKey findings:")
print(f"  LSA NDCG@10 = {comp['semantic'].get('ndcg_at_10',0):.6f} vs BM25 = {comp['bm25'].get('ndcg_at_10',0):.6f}")
print(f"  LSA query coverage significantly higher than BM25 (99.5% vs 54%)")
print(f"  Character n-grams handle synthetic vocabulary better than word-level regex")


Metric                   BM25        LSA      Delta
--------------------------------------------------
precision_at_10      0.380000   0.498000  +0.118000
recall_at_20         0.150625   0.193794  +0.043169
mrr_at_10            0.439917   0.516264  +0.076347
ndcg_at_10           0.297994   0.373320  +0.075326

Key findings:
  LSA NDCG@10 = 0.373320 vs BM25 = 0.297994
  LSA query coverage significantly higher than BM25 (99.5% vs 54%)
  Character n-grams handle synthetic vocabulary better than word-level regex


## 9. Linear Hybrid Fusion

### Formula: `0.5 × norm_bm25(d) + 0.5 × norm_sem(d)` with per-query min-max normalization.

In [9]:
hybrid_metrics = json.loads((OUTPUT_DIR / "hybrid" / "linear" / "metrics.json").read_text(encoding="utf-8"))
print(f"Linear Hybrid NDCG@10: {hybrid_metrics.get('ndcg_at_10', 'N/A')}")

ret_cmp = json.loads((OUTPUT_DIR / "comparison" / "retrieval_methods.json").read_text(encoding="utf-8"))

print(f"\n{'Metric':<18} {'BM25':>10} {'LSA':>10} {'Linear':>10}")
print("-" * 50)
for k in ["ndcg_at_10", "recall_at_20", "mrr_at_10"]:
    b = ret_cmp.get("bm25",{}).get(k,0)
    s = ret_cmp.get("semantic_lsa",{}).get(k,0)
    h = ret_cmp.get("hybrid_linear",{}).get(k,0)
    print(f"{k:<18} {b:>10.6f} {s:>10.6f} {h:>10.6f}")

print(f"\nLinear Hybrid improves over both single-channel methods on all metrics.")
print(f"Linear NDCG@10 = {ret_cmp['hybrid_linear'].get('ndcg_at_10',0):.6f}")


Linear Hybrid NDCG@10: 0.392327

Metric                   BM25        LSA     Linear
--------------------------------------------------
ndcg_at_10           0.297994   0.373320   0.392327
recall_at_20         0.150625   0.193794   0.200459
mrr_at_10            0.439917   0.516264   0.556833

Linear Hybrid improves over both single-channel methods on all metrics.
Linear NDCG@10 = 0.392327


## 10. User Profiles

Built from **training events only** using weighted event contributions and time decay.

In [10]:
from psr_srs_mvp.personalization import build_profiles

profiles = build_profiles(train_evts, {r["item_id"]: r for r in items_raw},
                          users_raw, cfg.event_weights, cfg.half_life_days)

warm = sum(1 for p in profiles.values() if p.profile_status == "warm")
cold = sum(1 for p in profiles.values() if p.profile_status == "cold_start")
no_pos = sum(1 for p in profiles.values() if p.profile_status == "no_positive")
no_hist = sum(1 for p in profiles.values() if p.profile_status in ("no_history","empty"))

print(f"Profile status distribution (100 users total):")
print(f"  Warm (has positive train events):  {warm}")
print(f"  Cold-start (flagged in users.csv): {cold}")
print(f"  No positive behavior:              {no_pos}")
print(f"  No history (zero sessions):        {no_hist}")
print(f"  Total:                             {warm+cold+no_pos+no_hist}")

# Example warm profile
warm_p = next((p for p in profiles.values() if p.profile_status == "warm"), None)
if warm_p:
    import json as _json
    print(f"\nExample warm user: {warm_p.user_id}")
    print(f"  Train events: {warm_p.train_event_count}")
    print(f"  Positive events: {warm_p.positive_event_count}")
    print(f"  Top categories: {_json.loads(warm_p.to_row().get('top_categories','[]'))}")
    print(f"  Top brands: {_json.loads(warm_p.to_row().get('top_brands','[]'))}")
    print(f"  Mean log price: {warm_p.mean_log_price}")


Profile status distribution (100 users total):
  Warm (has positive train events):  81
  Cold-start (flagged in users.csv): 4
  No positive behavior:              3
  No history (zero sessions):        12
  Total:                             100

Example warm user: user_000004
  Train events: 7
  Positive events: 2
  Top categories: ['Sports & Outdoors']
  Top brands: ['PeakAscent']
  Mean log price: 5.511523981601724


## 11. Personalized Re-ranking

### Formula: `0.70 × norm_retrieval + 0.12 × cat + 0.06 × subcat + 0.06 × brand + 0.06 × price`

In [11]:
per_metrics = json.loads((OUTPUT_DIR / "personalization" / "metrics.json").read_text(encoding="utf-8"))

print("Behavior NDCG comparison:")
for k in [5, 10, 20]:
    b = per_metrics.get(f"baseline_ndcg_at_{k}", 0)
    p = per_metrics.get(f"personalized_ndcg_at_{k}", 0)
    d = per_metrics.get(f"ndcg_at_{k}_delta", 0)
    print(f"  NDCG@{k:2d}: baseline={b:.6f}  personalized={p:.6f}  delta={d:+.6f}")

print(f"\nImproved:   {per_metrics.get('improved_request_count','N/A')}")
print(f"Unchanged:   {per_metrics.get('unchanged_request_count','N/A')}")
print(f"Worsened:    {per_metrics.get('worsened_request_count','N/A')}")
print(f"Sum:         {per_metrics.get('improved_plus_unchanged_plus_worsened','N/A')}")

# Show example improved request
pers_sr = list(csv.DictReader(open(OUTPUT_DIR / "personalization" / "search_results.csv",
                                    encoding="utf-8", newline="")))
req_metrics = list(csv.DictReader(open(OUTPUT_DIR / "personalization" / "request_metrics.csv",
                                        encoding="utf-8", newline="")))

# Find improved and unchanged requests
improved_req = [r for r in req_metrics if float(r.get("behavior_ndcg_at_10_delta", 0)) > 0.0001]
print(f"\nExample improved request:")
if improved_req:
    req = improved_req[0]
    rid = req["request_id"]
    print(f"  request_id: {rid}")
    print(f"  user_id: {req['user_id']}")
    print(f"  baseline_behavior_ndcg@10: {req['baseline_behavior_ndcg_at_10']}")
    print(f"  personalized_behavior_ndcg@10: {req['personalized_behavior_ndcg_at_10']}")
    print(f"  delta: {req['behavior_ndcg_at_10_delta']}")
    # Show Top-5 before/after
    req_results = [r for r in pers_sr if r["request_id"] == rid]
    print(f"\n  Rank  Item          PersonalScore  OrigRank  BehaviorGrade")
    for r in sorted(req_results, key=lambda r: int(r["rank"]))[:10]:
        ps = float(r["personalized_score"])
        o_r = r["original_rank"]
        bg = r["behavior_relevance_grade"]
        marker = " <<<" if bg != "0" else ""
        print(f"  {r['rank']:>4}  {r['item_id']:<12}  {ps:.6f}      {o_r:>3}        {bg}{marker}")


Behavior NDCG comparison:
  NDCG@ 5: baseline=0.008358  personalized=0.008358  delta=+0.000000
  NDCG@10: baseline=0.026146  personalized=0.026492  delta=+0.000345
  NDCG@20: baseline=0.030971  personalized=0.031204  delta=+0.000234

Improved:   2
Unchanged:   61
Worsened:    1
Sum:         64

Example improved request:
  request_id: req_000378
  user_id: user_000071
  baseline_behavior_ndcg@10: 0.301030
  personalized_behavior_ndcg@10: 0.333333
  delta: 0.032303

  Rank  Item          PersonalScore  OrigRank  BehaviorGrade
     1  item_000330   0.859003        2        0
     2  item_000250   0.851397        1        0
     3  item_000149   0.704598        3        0
     4  item_000292   0.635894        4        0
     5  item_000206   0.617853        5        0
     6  item_000281   0.587131        6        0
     7  item_000467   0.396958        9        1 <<<
     8  item_000396   0.389450        8        0
     9  item_000490   0.385670        7        0
    10  item_000063   0.2

## 12. Behavior Metrics

Comparing baseline (Linear Hybrid original order) vs personalized re-ranking on 64 evaluated test requests.

In [12]:
print(f"{'Behavior Metric':<22} {'Baseline':>10} {'Personalized':>14} {'Delta':>10}")
print("-" * 58)
for k in [5, 10, 20]:
    for m in ["hit_rate", "mrr", "ndcg", "positive_recall"]:
        b = per_metrics.get(f"baseline_{m}_at_{k}", 0)
        p = per_metrics.get(f"personalized_{m}_at_{k}", 0)
        d = per_metrics.get(f"{m}_at_{k}_delta", p - b)
        print(f"{m}_at_{k:<15} {b:>10.6f} {p:>14.6f} {d:>+10.6f}")

print(f"\nKey observation: Behavior metrics show minimal change.")
print(f"Reason: Candidate positive coverage is only 13.85% — most positive items")
print(f"are not in the Top-20 candidates that personalization can re-rank.")


Behavior Metric          Baseline   Personalized      Delta
----------------------------------------------------------
hit_rate_at_5                 0.031250       0.031250  +0.000000
mrr_at_5                 0.007812       0.007812  +0.000000
ndcg_at_5                 0.008358       0.008358  +0.000000
positive_recall_at_5                 0.020833       0.020833  +0.000000
hit_rate_at_10                0.109375       0.109375  +0.000000
mrr_at_10                0.016753       0.017529  +0.000775
ndcg_at_10                0.026146       0.026492  +0.000345
positive_recall_at_10                0.083333       0.083333  +0.000000
hit_rate_at_20                0.140625       0.140625  +0.000000
mrr_at_20                0.018837       0.019373  +0.000536
ndcg_at_20                0.030971       0.031204  +0.000234
positive_recall_at_20                0.109375       0.109375  +0.000000

Key observation: Behavior metrics show minimal change.
Reason: Candidate positive coverage is only 13.85% 

## 13. Qrels Protection Metrics

Verifying that personalization does not destroy retrieval relevance.

In [13]:
print("Qrels metrics (evaluated on same 64 test requests):")
print(f"{'Metric':<22} {'Baseline':>10} {'Personalized':>14} {'Delta':>10}")
print("-" * 58)
for k, m in [(10,"precision"), (20,"recall"), (10,"mrr"), (10,"ndcg")]:
    b = per_metrics.get(f"baseline_qrels_{m}_at_{k}", 0)
    p = per_metrics.get(f"personalized_qrels_{m}_at_{k}", 0)
    d = per_metrics.get(f"qrels_{m}_at_{k}_delta", p - b)
    print(f"qrels_{m}_at_{k:<11} {b:>10.6f} {p:>14.6f} {d:>+10.6f}")

print(f"\nPersonalized qrels NDCG@10 = {per_metrics.get('personalized_qrels_ndcg_at_10', 0):.6f}")
print(f"Qrels relevance is preserved — no degradation from personalization.")


Qrels metrics (evaluated on same 64 test requests):
Metric                   Baseline   Personalized      Delta
----------------------------------------------------------
qrels_precision_at_10            0.482812       0.490625  +0.007812
qrels_recall_at_20            0.000000       0.000000  +0.000000
qrels_mrr_at_10            0.496615       0.505208  +0.008594
qrels_ndcg_at_10            0.391907       0.396789  +0.004882

Personalized qrels NDCG@10 = 0.396789
Qrels relevance is preserved — no degradation from personalization.


## 14. Candidate Positive Coverage

**This is the key diagnostic.** Personalization can only re-rank items already
in the Linear Hybrid Top-20 candidates. Items not retrieved cannot be surfaced.

In [14]:
cov = {
    "eligible_positive_request_count": per_metrics.get("eligible_positive_request_count", 65),
    "covered_positive_request_count": per_metrics.get("covered_positive_request_count", 9),
    "request_level_candidate_positive_coverage": per_metrics.get("request_level_candidate_positive_coverage", 0.1385),
    "total_positive_item_count": per_metrics.get("total_positive_item_count", 143),
    "covered_positive_item_count": per_metrics.get("covered_positive_item_count", 17),
    "item_level_candidate_positive_recall": per_metrics.get("item_level_candidate_positive_recall", 0.1190),
}

print(f"Eligible test requests:          {cov['eligible_positive_request_count']}")
print(f"Requests with positive in Top-20: {cov['covered_positive_request_count']}")
print(f"Request-level coverage:          {cov['request_level_candidate_positive_coverage']:.4f}")
print(f"")
print(f"Total positive items (test):     {cov['total_positive_item_count']}")
print(f"Positive items in Top-20:        {cov['covered_positive_item_count']}")
print(f"Item-level recall:               {cov['item_level_candidate_positive_recall']:.4f}")

print(f"\nInterpretation:")
print(f"  Only {cov['request_level_candidate_positive_coverage']*100:.1f}% of test requests have")
print(f"  at least one positive item in their candidate set.")
print(f"  This is the PRIMARY reason behavior metric gains are minimal —")
print(f"  re-ranking cannot create new candidates, only re-order existing ones.")


Eligible test requests:          65
Requests with positive in Top-20: 9
Request-level coverage:          0.1385

Total positive items (test):     84
Positive items in Top-20:        10
Item-level recall:               0.1190

Interpretation:
  Only 13.8% of test requests have
  at least one positive item in their candidate set.
  This is the PRIMARY reason behavior metric gains are minimal —
  re-ranking cannot create new candidates, only re-order existing ones.


## 15. Fallback Verification

Cold-start/no-history users receive exact Linear Hybrid original order.

In [15]:
fb = {
    "fallback_user_count": per_metrics.get("fallback_user_count", 1),
    "fallback_request_count": per_metrics.get("fallback_request_count", 1),
    "fallback_exact_match_rate": per_metrics.get("fallback_exact_match_rate", 1.0),
}
print(f"Fallback users:                  {fb['fallback_user_count']}")
print(f"Fallback requests:               {fb['fallback_request_count']}")
print(f"Exact match rate:                {fb['fallback_exact_match_rate']:.4f}")

print(f"\nFallback reasons (multi-label):")
for reason in ["fallback_due_to_cold_start_flag", "fallback_due_to_no_train_session",
               "fallback_due_to_no_positive_behavior", "fallback_due_to_empty_profile"]:
    print(f"  {reason}: {per_metrics.get(reason, 'N/A')}")

print(f"\nAll fallback requests are verified to have exact item order match with baseline.")


Fallback users:                  1
Fallback requests:               1
Exact match rate:                1.0000

Fallback reasons (multi-label):
  fallback_due_to_cold_start_flag: 1
  fallback_due_to_no_train_session: 0
  fallback_due_to_no_positive_behavior: 0
  fallback_due_to_empty_profile: 1

All fallback requests are verified to have exact item order match with baseline.


## 16. Full Model Comparison

In [16]:
print(f"{'Stage':<20} {'NDCG@10':>10} {'Recall@20':>10}")
print("-" * 42)
print(f"{'BM25':<20} {bm25_metrics.get('ndcg_at_10', 0):>10.6f} {bm25_metrics.get('recall_at_20', 0):>10.6f}")
print(f"{'LSA':<20} {lsa_metrics.get('ndcg_at_10', 0):>10.6f} {lsa_metrics.get('recall_at_20', 0):>10.6f}")
print(f"{'Linear Hybrid':<20} {hybrid_metrics.get('ndcg_at_10', 0):>10.6f} {hybrid_metrics.get('recall_at_20', 0):>10.6f}")
print(f"{'Personalized (qrels)':<20} {per_metrics.get('personalized_qrels_ndcg_at_10', 0):>10.6f} {per_metrics.get('personalized_qrels_recall_at_20', 0):>10.6f}")


Stage                   NDCG@10  Recall@20
------------------------------------------
BM25                   0.297994   0.150625
LSA                    0.373320   0.193794
Linear Hybrid          0.392327   0.200459
Personalized (qrels)   0.396789   0.000000


## 17. Conclusions & Limitations

### Key Findings

1. **LSA > BM25**: Semantic retrieval (NDCG@10 0.3733) significantly
   outperforms keyword retrieval (0.2980) on synthetic e-commerce data

2. **Linear Hybrid improves further**: Fusion of BM25 + LSA achieves
   NDCG@10 0.3923

3. **Personalization preserves relevance**: Qrels NDCG@10 0.3968,
   no degradation from re-ranking

4. **Behavior gains are minimal**: NDCG@10 delta +0.0003 —
   candidate positive coverage is the bottleneck at 13.85%

5. **Fallback is correct**: 100% exact match for cold-start users

### Limitations

- **Synthetic data**: Rule-based simulation, not real user behavior
- **Small scale**: 500 items, 100 users, ~6k events
- **Candidate bottleneck**: Only 13.85% of eligible requests have
  positive items in Top-20 candidates
- **No online feedback**: No A/B test, no real latency measurement
- **Fixed weights**: All fusion and personalization weights are hand-set
- **No Learning to Rank**: Current personalization is rule-based affinity

### Next Steps (beyond MVP)

- Learning to Rank with behavioral training data
- Neural embedding models (Sentence-BERT) for semantic retrieval
- Online A/B testing framework
- Production deployment with real user data
